The TextRank algorithm works like a graph where the sentences or words in the text are represented by points (called nodes). These points are connected by lines (called edges), and each line has a weight that corresponds to the similarity between the connected sentences or words. This similarity is calculated by looking at how many words are shared between two sentences or groups of words, and these results are organized into a table (a matrix) that shows the degree of similarity between each pair.

- Retrieve data from the Reuters corpus

- Train a TxetRank model using this dataset


Testing TextRank with a financial article from Reuters

[Google unveils AI upgrades at I/O conference amid search challenges — Reuters (May 20, 2025)](https://www.reuters.com/business/google-unveil-ai-upgrades-io-conference-amid-search-challenges-2025-05-20/)

In [17]:
import spacy
import pytextrank
from rouge_score import rouge_scorer
from bert_score import score



text = """
MOUNTAIN VIEW, California, May 20 (Reuters) - Alphabet's (GOOGL.O), opens new tab Google said on Tuesday it would put artificial intelligence into the hands of more Web surfers while teasing a $249.99-a-month subscription for its AI power users, its latest effort to fend off growing competition from startups like OpenAI.
Google unveiled the plans amid a flurry of demos that included new smart glasses during its annual I/O conference in Mountain View, California, which has adopted a tone of increased urgency since the rise of generative AI challenged the tech company's longtime stronghold of organizing and retrieving information on the internet.
In recent months, Google has become more aggressive in asserting it has caught up to competitors after appearing flat-footed upon the release of Microsoft-backed (MSFT.O), opens new tab OpenAI's ChatGPT in 2022.
On Tuesday, it further laid out a vision for Google Search that lets consumers ask virtually anything, from simple queries to complex research questions, from analyzing what a smartphone camera sees to fetching an event ticket to buy.
Google likewise said it aims to build AI that is personal and proactive, whether phoning a store for users or sending students a practice test generated on the fly.
CEO Sundar Pichai said at the conference that Alphabet would build such AI with the cost in mind as well. "Over and over, we've been able to deliver the best models at the most effective price point," he said.
Google's AI assistant Gemini app now has more than 400 million monthly active users, Pichai said.
In a major update, the company said consumers across the United States now can switch Google Search into “AI Mode.”
Showcased in March, opens new tab as an experiment open to test users, the feature dispenses with the Web’s standard fare in favor of computer-generated answers for complicated queries.
Google also announced an “AI Ultra Plan,” which for $249.99 monthly provides users with higher limits on AI and early access to experimental tools like Project Mariner, an internet browser extension that can automate keystrokes and mouse clicks, and Deep Think, a version of its top-shelf Gemini model that is more capable of reasoning through complicated tasks.
The price is comparable to $200 monthly plans from AI model developers OpenAI and Anthropic, underscoring how companies are exploring ways to pay for the exorbitant price tag of AI development.
Google's new plan also includes 30 terabytes of cloud storage and an ad-free YouTube subscription. Google already offers other subscription options, including a $19.99-per-month service with access to some AI capabilities unavailable for most free users and cheaper plans with additional cloud storage. Last week, the company told Reuters it had signed up more than 150 million subscribers across those plans.
Pichai told reporters that the rise of generative AI was not at the full expense of online search.
This “feels very far from a zero-sum moment,” said Pichai. “The kind of use cases we are serving in search is dramatically expanding” because of AI.
Alphabet shares closed 1.5% lower at $165.32 on Tuesday. Google made a return to a bumpy effort years ago around smart glasses, demonstrating frames with its new Android XR software. Since its early efforts, rival Meta Platforms (META.O)
, opens new tab has brought its own glasses with AI to market.
On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' lenses.
Gemini, meanwhile, answered queries about one of the wearer's surroundings as she walked around Mountain View's Shoreline Amphitheater.
An XR headset being developed in partnership with Samsung (005930.KS)
, opens new tab will launch later this year, an official said. Google also announced new partnerships with glasses designers Warby Parker and Gentle Monster to develop headsets with Android XR. Earlier this month, Alphabet stock lost $150 billion in market value in one day after an Apple (AAPL.O)
, opens new tab executive testified during one of Google's antitrust cases that AI offerings had caused a decline in searches on Apple's Safari Web browser for the first time.
In turn, some analysts reassessed how to measure Google's dominant search market share, with one estimate stating it could fall to less than 50% from around 90% in five years. The analysts cited a behavioral shift drawing consumers toward AI chatbots where they once used traditional search engines.
However, Robby Stein, an executive on the search team, in an interview said allowing users to answer more challenging questions through AI could enable "new opportunities to create hyper-relevant, useful advertising." Ads make up the majority of Google's revenue.
Investment in AI accounts for most of Alphabet's $75 billion in forecasted capital expenditures this year, a dramatic uptick from the $52.5 billion in 2024 spending that the company reported.
Tuesday's announcements included further updates to Google's work to deliver a "universal AI agent," which can perform tasks on someone’s behalf without additional prompting.
In a number of demos, Google drew on capabilities developed in a testing ground it has called Project Astra to show off what its latest AI could do.
These included pointing a smartphone camera at a written invitation and having AI add the event to a user’s calendar.
Google also presented a new AI model called Veo 3 that generates video and audio to create more realistic film snippets for creators.
"""
# load a spaCy model, depending on language, scale, etc.
nlp = spacy.load("en_core_web_sm")

# add PyTextRank to the spaCy pipeline
nlp.add_pipe("textrank")
doc = nlp(text)

generated_sentences = [sent.text for sent in doc._.textrank.summary(limit_phrases=10, limit_sentences=3)]
generated_summary = " ".join(generated_sentences)

print("\n--- Summary ---\n")
print(generated_summary)


--- Summary ---

The price is comparable to $200 monthly plans from AI model developers OpenAI and Anthropic, underscoring how companies are exploring ways to pay for the exorbitant price tag of AI development.
 Google unveiled the plans amid a flurry of demos that included new smart glasses during its annual I/O conference in Mountain View, California, which has adopted a tone of increased urgency since the rise of generative AI challenged the tech company's longtime stronghold of organizing and retrieving information on the internet.
 Pichai told reporters that the rise of generative AI was not at the full expense of online search.



In [18]:
benchmark_summary = (
    "Google brings 'AI Mode' to all U.S. users in bid to maintain search dominance. "
    "Analysts predict Google's search market share may drop below 50% in five years. "
    'Google touts glasses, new AI features as step towards "universal" assistants.'
)

'''
print("\n--- Summary generated ---\n")
print(generated_summary)

print("\n--- Summary benchmark---\n")
print(benchmark_summary)
'''

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(benchmark_summary, generated_summary)

print("\n--- Scores ROUGE (compared to benchmark summary) ---")
for key, value in scores.items():
    print(f"{key}: Precision={value.precision:.3f} Recall={value.recall:.3f} ")





--- Scores ROUGE (compared to benchmark summary) ---
rouge1: Precision=0.098 Recall=0.250 
rouge2: Precision=0.000 Recall=0.000 
rougeL: Precision=0.049 Recall=0.125 


In [19]:
#Compute BERTScore
P, R, F1 = score([generated_summary], [benchmark_summary], lang="en", verbose=True)

print("\n--- Scores BERTscore (compared to benchmark summary) ---")
print(f"Precision: {P.mean():.4f}")
print(f"Recall: {R.mean():.4f}")
print(f"F1: {F1.mean():.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:17<00:00, 17.33s/it]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00,  5.04it/s]

done in 17.55 seconds, 0.06 sentences/sec

--- Scores BERTscore (compared to benchmark summary) ---
Precision: 0.8242
Recall: 0.8485
F1: 0.8362


The summary generated by TextRank manages to capture two of the main ideas present in the reference summary, such as the cost of AI services and Google’s efforts to respond to competition in the search domain. However, the ROUGE scores are very low, notably a ROUGE-2 of 0.000, which indicates that no bigrams (pairs of consecutive words) are shared between the two summaries. This highlights a limitation of automatic evaluation: ROUGE mainly measures lexical overlap without accounting for semantic similarity. In other words, even if the meaning is partially preserved, ROUGE may give the impression that the summary is of poor quality, which is not always accurate.

To better evaluate semantic similarity, we also used BERTScore, which compares embeddings generated by a pre-trained language model rather than relying on exact word matches. In this case, the BERTScore results are significantly more favorable, with a Precision of 0.8242, Recall of 0.8485, and F1-score of 0.8362. These values suggest that the generated summary conveys much of the same meaning as the reference, even though the wording is different. This confirms that BERTScore can provide a more nuanced assessment of summary quality by focusing on meaning rather than surface form.